<a href="https://colab.research.google.com/github/lucy7noh/LLM_Langgraph_study/blob/chatbot_dev/week03_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [139]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [140]:
# 뉴스 기사 6개
기사1 = "삼성전자가 신형 갤럭시 스마트폰을 출시했습니다. AI 카메라 기능과 배터리 성능이 향상됐습니다."
기사2 = "애플이 아이폰 신제품을 발표했습니다. 최신 칩셋과 카메라를 개선했으며 배터리 수명도 늘었습니다."
기사3 = "국내 프로야구 한국시리즈가 개막했습니다. 투수진과 타선의 조화가 우승의 핵심 요소입니다."
기사4 = "국가대표 축구팀이 월드컵 예선에서 승리했습니다. 공격수의 결승골이 팀을 이끌었습니다."
기사5 = "정부가 인공지능 산업 육성을 위한 투자 계획을 발표했습니다. AI 반도체 개발도 지원합니다."
기사6 = "국내 스타트업이 AI 기반 의료 진단 서비스를 출시했습니다. 인공지능으로 암 조기 발견율을 높입니다."

print("기사1 [스마트폰]:", 기사1[:40], "...")
print("기사2 [스마트폰]:", 기사2[:40], "...")
print("기사3 [스포츠]  :", 기사3[:40], "...")
print("기사4 [스포츠]  :", 기사4[:40], "...")
print("기사5 [AI]     :", 기사5[:40], "...")
print("기사6 [AI]     :", 기사6[:40], "...")

기사1 [스마트폰]: 삼성전자가 신형 갤럭시 스마트폰을 출시했습니다. AI 카메라 기능과 배터 ...
기사2 [스마트폰]: 애플이 아이폰 신제품을 발표했습니다. 최신 칩셋과 카메라를 개선했으며 배 ...
기사3 [스포츠]  : 국내 프로야구 한국시리즈가 개막했습니다. 투수진과 타선의 조화가 우승의  ...
기사4 [스포츠]  : 국가대표 축구팀이 월드컵 예선에서 승리했습니다. 공격수의 결승골이 팀을  ...
기사5 [AI]     : 정부가 인공지능 산업 육성을 위한 투자 계획을 발표했습니다. AI 반도체 ...
기사6 [AI]     : 국내 스타트업이 AI 기반 의료 진단 서비스를 출시했습니다. 인공지능으로 ...


In [141]:
import re
from konlpy.tag import Okt

okt = Okt()

불용어_목록 = [
    '이', '그', '저', '것', '들', '의', '를', '에', '가', '은', '는', '이', '가',
    '향상', '최신', '핵심', '지원', '정부', '포함', '출시', '발표', '계획',
    '기능', '개선', '신제품', '육성', '산업', '조기', '서비스', '대폭',
    '성능', '수명', '조화', '요소', '분석', '이끌', '승리', '계획', '개발', '기반',
    '발견', '높이', '신형', '새로운', '크게', '결승골', '국가대표', '스타트업',
    '칩셋', '진과', '공격수', '투자', '선의'
]

def 단어_정규화(명사_리스트):
    변환사전 = {
        '전자': '삼성',
        '프로야구': '야구',
        '축구팀': '축구',
        '시리즈': '한국시리즈'
    }

    결과 = []
    for 단어 in 명사_리스트:
        결과.append(변환사전.get(단어, 단어))
    return 결과

def 정제(문장):
    문장 = re.sub(r'http\S+|www\S+', ' ', 문장)
    문장 = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', 문장)
    문장 = re.sub(r'\s+', ' ', 문장).strip()
    return 문장

def 합성명사_복원(명사_리스트):
    결과 = []
    i = 0

    while i < len(명사_리스트):
        if i < len(명사_리스트) - 1 and 명사_리스트[i] == '인공' and 명사_리스트[i+1] == '지능':
            결과.append('인공지능')
            i += 2
        else:
            결과.append(명사_리스트[i])
            i += 1

    return 결과

def 전처리(기사):
    # 1. AI 표현 통일
    기사 = 기사.replace("AI", "AI 인공지능")

    # 2. 정제
    기사 = 정제(기사)

    # 3. 명사 추출
    명사_리스트 = okt.nouns(기사)

    # 4. 합성명사 복원
    명사_리스트 = 합성명사_복원(명사_리스트)

    # 5. 표현 통일
    명사_리스트 = 단어_정규화(명사_리스트)

    # 6. 불용어 제거 + 2글자 이상
    정제된_명사 = [
        단어 for 단어 in 명사_리스트
        if 단어 not in 불용어_목록 and len(단어) >= 2
    ]
    return 정제된_명사

def 전처리_파이프라인(기사):
    return 전처리(기사)

처리_기사1 = 전처리(기사1)
처리_기사2 = 전처리(기사2)
처리_기사3 = 전처리(기사3)
처리_기사4 = 전처리(기사4)
처리_기사5 = 전처리(기사5)
처리_기사6 = 전처리(기사6)

print("전처리 결과 (명사만 추출):")
print(f"  기사1: {처리_기사1}")
print(f"  기사2: {처리_기사2}")
print(f"  기사3: {처리_기사3}")
print(f"  기사4: {처리_기사4}")
print(f"  기사5: {처리_기사5}")
print(f"  기사6: {처리_기사6}")

전처리 결과 (명사만 추출):
  기사1: ['삼성', '갤럭시', '스마트폰', '인공지능', '카메라', '배터리']
  기사2: ['애플', '아이폰', '카메라', '배터리']
  기사3: ['국내', '야구', '한국', '한국시리즈', '개막', '투수', '우승']
  기사4: ['축구', '월드컵', '예선']
  기사5: ['인공지능', '인공지능', '반도체']
  기사6: ['국내', '인공지능', '의료', '진단', '인공지능']


In [142]:
문서목록 = [' '.join(tokens) for tokens in 전처리결과]

vectorizer = TfidfVectorizer(norm=None)
tfidf_matrix = vectorizer.fit_transform(문서목록)
feature_names = vectorizer.get_feature_names_out()

print("STEP 2. TF-IDF 행렬 크기")
print(tfidf_matrix.shape)
print()

STEP 2. TF-IDF 행렬 크기
(6, 21)



In [143]:
def 핵심키워드_top3(tfidf_matrix, feature_names, 기사번호):
    row = tfidf_matrix[기사번호].toarray().flatten()
    인덱스 = row.argsort()[::-1]
    top_indices = [idx for idx in 인덱스 if row[idx] > 0][:3]
    return [feature_names[idx] for idx in top_indices]

라벨 = ["삼성 스마트폰", "애플 스마트폰", "야구", "축구", "AI 정책", "AI 의료"]

for i in range(len(기사목록)):
    top3 = 핵심키워드_top3(tfidf_matrix, feature_names, i)
    print(f"기사{i+1} 키워드: {top3}  ← {라벨[i]} 기사")

기사1 키워드: ['삼성', '갤럭시', '스마트폰']  ← 삼성 스마트폰 기사
기사2 키워드: ['아이폰', '애플', '배터리']  ← 애플 스마트폰 기사
기사3 키워드: ['투수', '선의', '시리즈']  ← 야구 기사
기사4 키워드: ['축구', '예선', '월드컵']  ← 축구 기사
기사5 키워드: ['인공지능', '반도체']  ← AI 정책 기사
기사6 키워드: ['인공지능', '진단', '의료']  ← AI 의료 기사


In [144]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("STEP 4. 기사 유사도 행렬")
header = "       " + "  ".join([f"기사{i+1}" for i in range(len(기사목록))])
print(header)

for i in range(len(기사목록)):
    row_str = "  ".join([f"{similarity_matrix[i][j]:.3f}" for j in range(len(기사목록))])
    print(f"기사{i+1}  [{row_str}]")
print()

STEP 4. 기사 유사도 행렬
       기사1  기사2  기사3  기사4  기사5  기사6
기사1  [1.000  0.335  0.000  0.000  0.256  0.204]
기사2  [0.335  1.000  0.000  0.000  0.000  0.000]
기사3  [0.000  0.000  1.000  0.000  0.000  0.122]
기사4  [0.000  0.000  0.000  1.000  0.000  0.000]
기사5  [0.256  0.000  0.000  0.000  1.000  0.524]
기사6  [0.204  0.000  0.122  0.000  0.524  1.000]



In [145]:
pairs = []
n = len(기사목록)

for i in range(n):
    for j in range(i + 1, n):
        pairs.append((i + 1, j + 1, similarity_matrix[i][j]))

pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)

print("STEP 5. 가장 비슷한 기사 쌍 TOP 3")
for a, b, score in pairs_sorted[:3]:
    print(f"기사{a} ↔ 기사{b}: {score:.3f}")

STEP 5. 가장 비슷한 기사 쌍 TOP 3
기사5 ↔ 기사6: 0.524
기사1 ↔ 기사2: 0.335
기사1 ↔ 기사5: 0.256
